<a href="https://colab.research.google.com/github/chight1/PaddlePaddle-Practice/blob/main/HousingPrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [112]:
!pip install paddlepaddle
import pandas as pd
from sklearn.preprocessing import StandardScaler

X_train = pd.read_csv('train.csv')
y_train = X_train['SalePrice']
X_train = X_train.drop(['SalePrice'], axis=1)

X_test = pd.read_csv('test.csv')

features_to_drop = ['Alley', 'FireplaceQu', 'PoolQC', 'MiscFeature', 'Fence', 'MasVnrType', 'LotFrontage', 'GarageQual', 'GarageCond', 'BsmtCond']
X_train = X_train.drop(features_to_drop, axis=1)
X_test = X_test.drop(features_to_drop, axis=1)

cat_features = ['BsmtQual', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish']
X_train[cat_features] = X_train[cat_features].fillna('NA')
X_test[cat_features] = X_test[cat_features].fillna('NA')

median_masvnr = X_train['MasVnrArea'].median()
median_garageyr = X_train['GarageYrBlt'].median()
X_train['MasVnrArea'] = X_train['MasVnrArea'].fillna(median_masvnr)
X_train['GarageYrBlt'] = X_train['GarageYrBlt'].fillna(median_garageyr)
X_test['MasVnrArea'] = X_test['MasVnrArea'].fillna(median_masvnr)
X_test['GarageYrBlt'] = X_test['GarageYrBlt'].fillna(median_garageyr)

mode_electrical = X_train['Electrical'].mode()[0]
X_train['Electrical'] = X_train['Electrical'].fillna(mode_electrical) ## only one null
X_test['Electrical'] = X_test['Electrical'].fillna(mode_electrical)

for col in ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']:
    X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

for col in ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
            'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']:
    X_test[col] = X_test[col].fillna(X_train[col].median())

# get null counts of each column
null_counts = X_test.isnull().sum()
null_counts[null_counts > 0]

# scaling
scaler = StandardScaler()
numeric_cols = [
    'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
    'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal'
]
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

#onehot
print("Features before:", X_train.shape[1])
print("Features before:", X_test.shape[1])
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
#keep only columns they have in common
common_cols = X_train.columns.intersection(X_test.columns)
X_train = X_train[common_cols]
X_test = X_test[common_cols]
print("Features after:", X_train.shape[1])
print("Features after:", X_test.shape[1])

# recombine x and y to get corr mat
# corr_df = X_train.copy()
# corr_df['SalePrice'] = y_train
# corr_matrix = corr_df.corr().abs()

# Keep features with correlation >= 0.3 to SalePrice
# strong_corr = corr_matrix['SalePrice'][corr_matrix['SalePrice'] >= 0.3].index.drop('SalePrice')

# Drop all weak corr features
# X_train_reduced = X_train[strong_corr]
# X_test_reduced = X_test[strong_corr]
X_train_reduced = X_train
X_test_reduced = X_test

print("Original features:", X_train.shape[1])
print("Reduced features:", X_train_reduced.shape[1])
print("Original features:", X_test.shape[1])
print("Reduced features:", X_test_reduced.shape[1])

Features before: 70
Features before: 70
Features after: 243
Features after: 243
Original features: 243
Reduced features: 243
Original features: 243
Reduced features: 243


In [113]:
from sklearn.linear_model import LinearRegression
import numpy as np

y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train_reduced, y_train_log)

# Predict
y_pred_log = model.predict(X_test_reduced)

# Convert back to original scale
y_pred = np.expm1(y_pred_log)

# Make sure it isn't garbage
print(pd.Series(y_pred).describe())


count      1459.000000
mean     176571.622136
std       79933.836809
min       34695.733615
25%      125020.396690
50%      155934.842390
75%      203655.014124
max      788263.400355
dtype: float64


In [114]:
# now do test with sklearn
y_pred_log = model.predict(X_test_reduced)
y_pred = np.expm1(y_pred_log)

# make sure it isn't garbage
print(pd.Series(y_pred).describe())

original_test = pd.read_csv('test.csv')

submission_df = pd.DataFrame({
    'Id': original_test['Id'].values,
    'SalePrice': y_pred
})

submission_df.to_csv('submission_sklearn.csv', index=False)


count      1459.000000
mean     176571.622136
std       79933.836809
min       34695.733615
25%      125020.396690
50%      155934.842390
75%      203655.014124
max      788263.400355
dtype: float64
